# GGIT LNG Terminals — summary sheets, **Asia only**, Q4 2026 data

An Asia-scoped cut of the eleven LNG summary tabs, built for a South/Southeast Asia
gas write-up. Forked from `../2026-q4-lng-terminals/GGIT-LNG-summary-sheets-2026Q4.ipynb`
— the method, and every caveat attached to it, is documented there and in that folder's
`README.md`. **Read those first.**

**The one structural change is where the geographic filter lands.** Scope is applied at
*pivot* time, never at load time. The capex regional cost rates are means over a
**global** sample of terminals, so filtering the input to Asia before they are computed
would silently change every estimated cost cell. The full world dataset is therefore
carried all the way through the fallback, the owner split and the capex rate fit, and
`SCOPE` is applied only when a tab is finally pivoted.

This is an analysis cut, not a release artifact — nothing here is published, and the
notebook never writes to Google Sheets.

In [1]:
%pip install -q -e ../../../gem-tracker-constants

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## Configuration

`DATA_CSV` is the full all-fields database download (2026-08-28), which is a superset of
the distributed LNG export — same 1,281 rows, 115 columns instead of 73. Both carry every
column this notebook needs; the all-fields file is used because it also keeps `Substatus`
and the `[ref]` columns for spot-checking a number by hand.

`SCOPE` is the geographic filter. It is a mask over the working frame, applied per tab.

In [3]:
RELEASE_LABEL = "Q4 2026 — Asia"

# Full all-fields database download (superset of the distributed lng export).
DATA_CSV = Path("all-fields-2026-08-28T161944.csv")

# No published Asia tabs exist to validate against - this is an analysis cut.
REFERENCE_JSON = Path("published-2026-q4-reference.json")

OUTPUT_XLSX = Path("GGIT-LNG-summary-sheets-Asia-2026Q4.xlsx")

# ---------------------------------------------------------------------------
# SCOPE - the only structural difference from the global notebook.
# Applied at PIVOT time (see the scope section), never at load time, so that the
# capex regional cost means stay fitted on the full global sample.
# ---------------------------------------------------------------------------
SCOPE_NAME = "Asia"
SCOPE_REGIONS = ["Asia"]          # top-level Region values kept in every tab

# The write-up's actual subject: UN M49 "Southern Asia" + "South-eastern Asia".
# Used only by the dedicated South/Southeast section at the end.
SSEA_SUBREGIONS = ["Southern Asia", "South-eastern Asia"]

FUEL = "LNG"  # excludes the NH3 / LH2 / eLNG / Oil rows that share the tracker

# Status order as published. The tracker stores these lowercase.
STATUS_ORDER = ["proposed", "construction", "shelved", "cancelled",
                "operating", "idled", "mothballed", "retired"]
STATUS_LABEL = {s: s.capitalize() for s in STATUS_ORDER}

# Statuses counted as built capacity in the start-year tab. NOTE: the published tab is
# titled "Operating LNG capacity by start year" but the pivot filter includes idled,
# mothballed and retired. The title is wrong, not the data - verified against the pivot spec.
BUILT_STATUSES = ["operating", "idled", "mothballed", "retired"]
START_YEAR_FLOOR = 1980  # pivot's row floor; excludes pre-1980 capacity

# --- capex tunables (from the retired 2023 cost notebook) ---
CAPEX_QLO, CAPEX_QHI = 0.10, 0.90   # quantile trim on cost-per-mtpa sample
CAPEX_MIN_POINTS = 3                # regional mean needs >= this many points, else global

# --- the floating-unit costing defect ---------------------------------------
# The retired source notebook tests `Floating == "yes"` while the column holds True/blank,
# so its floating branch NEVER fires: every unit, FSRU included, is costed at its onshore
# regional rate. In this download `Floating` holds Python True on 350 of 1,257 LNG rows,
# and 70 Asian import units (214.9 mtpa) carry no reported cost, so the estimate is what
# actually prices them. FSRUs run roughly half the cost per mtpa of onshore regas
# (Asia import: 145.2 vs 280.9 US$m/mtpa), so the defect systematically OVERSTATES capex
# in exactly the FSRU-heavy regions this cut is about.
#
#   True  -> fixed: floating units cost at the offshore rate  (this cut's default)
#   False -> reproduces the published global tabs, defect included
CAPEX_FIX_FLOATING = True

FLOATING_TRUE_TOKEN = True if CAPEX_FIX_FLOATING else "yes"
# The source also assigned the GLOBAL offshore mean unconditionally, so regional offshore
# means were never applied. Fixing the token alone leaves that in place; the full fix
# treats offshore like onshore - regional mean, global fallback under CAPEX_MIN_POINTS.
CAPEX_FLOATING_ALWAYS_GLOBAL = not CAPEX_FIX_FLOATING

## `gem-tracker-constants` cross-check

The package's `TERMINAL_STATUS` disagrees with both the tracker and the published tables:
it says `Idle`, they say `Idled`. Flagged rather than silently worked around — the fix
belongs in the package's `statuses.yaml`, not here.

In [4]:
try:
    from gem_tracker_constants import TERMINAL_STATUS
    pkg = {s.lower() for s in TERMINAL_STATUS}
    here = set(STATUS_ORDER)
    if pkg != here:
        print("gem-tracker-constants TERMINAL_STATUS disagrees with this release:")
        print(f"  in package, not in data: {sorted(pkg - here)}")
        print(f"  in data, not in package: {sorted(here - pkg)}")
        print("  -> known: statuses.yaml says 'Idle', tracker + published tables say 'Idled'.")
    else:
        print("TERMINAL_STATUS matches.")
except ImportError:
    print("gem-tracker-constants not installed - skipping cross-check.")

gem-tracker-constants TERMINAL_STATUS disagrees with this release:
  in package, not in data: ['idle']
  in data, not in package: ['idled']
  -> known: statuses.yaml says 'Idle', tracker + published tables say 'Idled'.


## Load and filter

One row per **unit**, not per terminal — an export terminal with three trains is three rows.
Every published tab sums unit capacity, so unit grain is correct throughout; the only place
terminal grain matters is the capex cost sample, which dedupes on `ProjectID`.

In [5]:
NUMERIC_COLS = ["Capacity", "CapacityinMtpa", "CostUSD",
                "TotKnownTerminalCostsUSD", "ActualStartYear"]

# utf-8-sig: the gem-db-ops export carries a BOM, which otherwise lands in the
# first column name. Its id column is TerminalID where the 2025 download said
# ProjectID; normalise so the fallback and capex steps group on one name.
raw = pd.read_csv(DATA_CSV, encoding="utf-8-sig", low_memory=False)
if "ProjectID" not in raw.columns and "TerminalID" in raw.columns:
    raw = raw.rename(columns={"TerminalID": "ProjectID"})
    print("renamed TerminalID -> ProjectID")

REQUIRED = ["ProjectID", "UnitID", "Country/Area", "FacilityType", "Fuel", "Status",
            "Parent", "CapacityinMtpa", "CostUSD", "TotKnownTerminalCostsUSD",
            "ActualStartYear", "Region", "SubRegion", "Floating",
            "TotImportLNGTerminalCapacityinMtpa", "TotExportLNGTerminalCapacityinMtpa"]
missing = [c for c in REQUIRED if c not in raw.columns]
assert not missing, f"input is missing required columns: {missing}"
for c in NUMERIC_COLS:
    if c in raw.columns:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

# Trim the whitespace that survives a Sheets paste. Element-wise because all-NaN
# object columns have no .str accessor.
for c in raw.select_dtypes("object").columns:
    raw[c] = raw[c].map(lambda v: v.strip() if isinstance(v, str) else v)

lng = raw[raw["Fuel"] == FUEL].copy()
lng["is_floating"] = lng["Floating"].notna()   # download encodes True / blank, not yes / ''

# The tracker carries in-progress "[TO BE DELETED]" markers inside parent names
# (37 rows). The published owner tabs read clean names from the Sheet backend's
# owners tab, so strip the marker here or those parents split into two rows.
DELETED_MARKER = re.compile(r"\s*\[TO BE DELETED\]\s*")
n_marked = lng["Parent"].fillna("").str.contains(DELETED_MARKER).sum()
lng["Parent"] = lng["Parent"].map(
    lambda v: DELETED_MARKER.sub(" ", v).strip() if isinstance(v, str) else v)
print(f"stripped '[TO BE DELETED]' from {n_marked} Parent values")

print(f"{len(raw):,} rows in download -> {len(lng):,} {FUEL} rows")
print("\nFacilityType:"); print(lng["FacilityType"].value_counts(dropna=False))
print("\nStatus:"); print(lng["Status"].value_counts(dropna=False))

unknown = set(lng["Status"].dropna()) - set(STATUS_ORDER)
assert not unknown, f"unexpected statuses: {unknown}"

renamed TerminalID -> ProjectID
stripped '[TO BE DELETED]' from 67 Parent values
1,281 rows in download -> 1,257 LNG rows

FacilityType:
FacilityType
import    775
export    480
NaN         2
Name: count, dtype: int64

Status:
Status
operating       399
cancelled       337
proposed        247
construction    111
shelved         100
retired          30
mothballed       15
idled            14
NaN               4
Name: count, dtype: int64


## Terminal-level capacity fallback

Some terminals carry no unit-level `CapacityinMtpa` at all — every unit is blank — but do
carry a terminal total in `TotExportLNGTerminalCapacityinMtpa` /
`TotImportLNGTerminalCapacityinMtpa`, repeated on every unit row. Commonwealth LNG is the
example: six proposed trains, all blank, with 9.5 mtpa on the terminal.

**The rule: where unit capacity is missing, take the terminal total once** — once per
terminal, not once per unit. This is what produces the `=353.12+9.5` in the published
Northern America cell.

Two things make this safe to apply generally:

- It only fires when **every** unit on a terminal is blank. In all 22 terminals where only
  *some* units are blank, the unit sum already equals the terminal total, so there is
  nothing to add.
- Both affected terminals also have no unit-level `CostUSD`, only
  `TotKnownTerminalCostsUSD`, so the same synthetic row carries the terminal cost. Without
  it an $11bn proposed terminal contributes zero capex.

This replaces the hard-coded adjustment deltas an earlier draft of this notebook carried.
A rule generalises to the 2026 run; five hand-entered numbers do not.

In [6]:
TERMINAL_TOTAL_COL = {
    "export": "TotExportLNGTerminalCapacityinMtpa",
    "import": "TotImportLNGTerminalCapacityinMtpa",
}

def apply_capacity_fallback(frame):
    """Where every unit on a terminal lacks CapacityinMtpa, append one synthetic row
    carrying the terminal total (and the terminal cost, if no unit carries one)."""
    frame = frame.copy()
    frame["FromTerminalTotal"] = False       # capex excludes these; see the capex section
    extra = []
    for (pid, ft), g in frame.groupby(["ProjectID", "FacilityType"]):
        if ft not in TERMINAL_TOTAL_COL:
            continue
        if g["CapacityinMtpa"].notna().any():
            continue                                    # normal unit-level sum applies
        totals = g[TERMINAL_TOTAL_COL[ft]].dropna().unique()
        if len(totals) == 0 or totals[0] == 0:
            continue                                    # nothing to fall back to
        row = g.iloc[0].copy()
        row["CapacityinMtpa"] = float(totals[0])
        row["UnitName"] = f"{g['TerminalName'].iloc[0]} (terminal total)"
        row["FromTerminalTotal"] = True
        if g["CostUSD"].isna().all():
            row["CostUSD"] = row["TotKnownTerminalCostsUSD"]
        extra.append(row)
        print(f"  fallback: {g['TerminalName'].iloc[0]} ({ft}) "
              f"{len(g)} blank unit(s) -> {totals[0]} mtpa")
    if not extra:
        return frame
    return pd.concat([frame, pd.DataFrame(extra)], ignore_index=True)

print("applying terminal-level capacity fallback:")
lng = apply_capacity_fallback(lng)
print(f"-> {len(lng):,} rows")

applying terminal-level capacity fallback:
  fallback: Delfin FLNG Terminal (export) 3 blank unit(s) -> 13.2 mtpa
  fallback: Tianjin LNG Terminal (Beijing Gas Group) (import) 3 blank unit(s) -> 5.0 mtpa
  fallback: Tabeer LNG Terminal (import) 2 blank unit(s) -> 5.7 mtpa
-> 1,260 rows


## Scope

`in_scope()` is the geographic filter, and it is applied **per tab**, not at load. Every
step before this point — the LNG fuel filter, the terminal-capacity fallback, the parent
ownership split, and above all the capex regional rate fit — runs on the full world
dataset, which is what keeps the capex rates comparable with the published global tabs.

In [7]:
def in_scope(frame):
    """The geographic filter. Applied per tab, never at load - see the notes at the end."""
    return frame[frame["Region"].isin(SCOPE_REGIONS)]

_s = in_scope(lng)
print(f"scope = {SCOPE_NAME}: {len(_s):,} of {len(lng):,} {FUEL} unit rows, "
      f"{_s['ProjectID'].nunique():,} terminals, {_s['Country/Area'].nunique()} countries/areas")
print("\nsubregions in scope:")
print(_s.groupby("SubRegion")["CapacityinMtpa"].agg(["count", "sum"]).round(1).to_string())

scope = Asia: 528 of 1,260 LNG unit rows, 361 terminals, 34 countries/areas

subregions in scope:
                    count    sum
SubRegion                       
Central Asia            1    0.2
Eastern Asia          230  974.6
South-eastern Asia    112  318.9
Southern Asia          70  353.0
Western Asia           51  349.8


## Capacity tabs

Four tabs: {export, import} x {by region, by country/area}. Straight sum of
`CapacityinMtpa` by row label and status, matching the pivot specs
(rows=`SubRegion` or `Country/Area`, cols=`Status`, values=SUM `CapacityinMtpa`,
filters `FacilityType` and `Fuel`).

`In Development` is derived after adjustments, so it cannot desync the way the published
Northern America cell did.

In [8]:
IN_DEV = "In Development (Proposed + Construction)"

def capacity_pivot(facility_type, row_col):
    sub = in_scope(lng)                       # <-- scope applied here, post-fallback
    sub = sub[sub["FacilityType"] == facility_type]
    p = (sub.pivot_table(index=row_col, columns="Status",
                         values="CapacityinMtpa", aggfunc="sum")
            .reindex(columns=STATUS_ORDER))
    return p.fillna(0.0)

def finish_capacity(p, tab, index_name):
    """Apply adjustments, derive In Development, order columns, append Total."""
    out = pd.DataFrame(index=p.index)
    out["Proposed"] = p["proposed"]
    out["Construction"] = p["construction"]
    out[IN_DEV] = p["proposed"] + p["construction"]
    for s in ["shelved", "cancelled", "operating", "idled", "mothballed", "retired"]:
        out[STATUS_LABEL[s]] = p[s]
    out.index.name = index_name
    out = out.sort_index()
    # Drop rows that are zero across every status.
    out = out[out.abs().sum(axis=1) > 0]
    out.loc["Total"] = out.sum()
    return out.round(4)

capacity = {}
for ft, tab in [("export", "LNG export capacity by country/area"),
                ("import", "LNG import capacity by country/area")]:
    capacity[tab] = finish_capacity(capacity_pivot(ft, "Country/Area"), tab, "Country/Area")

for ft, tab in [("export", "LNG export capacity by region"),
                ("import", "LNG import capacity by region")]:
    capacity[tab] = finish_capacity(capacity_pivot(ft, "SubRegion"), tab, "Subregion")

print(f"--- {SCOPE_NAME}: LNG import capacity by subregion (mtpa) ---")
capacity["LNG import capacity by region"]

--- Asia: LNG import capacity by subregion (mtpa) ---


,Proposed,Construction,In Development (Proposed + Construction),Shelved,Cancelled,Operating,Idled,Mothballed,Retired
Subregion,,,,,,,,,
Eastern Asia,163.95,134.83,298.78,26.00,75.40,569.12,0.0,0.0,2.85
South-eastern Asia,51.84,18.10,69.94,23.24,54.96,62.60,0.0,0.0,0.10
Southern Asia,89.86,11.00,100.86,12.60,107.36,75.90,0.0,0.0,0.00
Western Asia,17.20,5.52,22.72,2.00,20.17,74.40,17.1,5.8,0.00
Total,322.85,169.45,492.30,63.84,257.89,782.02,17.1,5.8,2.95


### Region -> subregion scaffold

The pivots emit **subregions only**. The `Region` column and the empty-subregion rows in the
published tabs (Eastern Asia, Southern Europe, Micronesia and others carry no LNG capacity)
were added by hand at paste time. Rebuilt here from the download's own
`Region` / `SubRegion` pairing so the 2026 run does not need the same manual step.

In [9]:
SUBREGION_TO_REGION = (lng[["Region", "SubRegion"]].dropna().drop_duplicates()
                          .set_index("SubRegion")["Region"].to_dict())

def add_region_column(frame, label="Subregion"):
    """Prepend a Region column, blanked after the first row of each group (as published)."""
    body = frame.drop(index="Total").copy()
    body.index.name = label
    body = body.reset_index()
    body.insert(0, "Region", [SUBREGION_TO_REGION.get(s, "") for s in body[label]])
    body = body.sort_values(["Region", label], kind="stable").reset_index(drop=True)

    shown, prev = [], object()
    for r in body["Region"]:
        shown.append("" if r == prev else r)
        prev = r
    body["Region"] = shown

    total = frame.loc[["Total"]].copy()
    total.index.name = label
    total = total.reset_index()
    total.insert(0, "Region", "")

    return pd.concat([body, total], ignore_index=True)

region_tabs = {t: add_region_column(capacity[t])
               for t in ["LNG export capacity by region", "LNG import capacity by region"]}
region_tabs["LNG export capacity by region"].head(4)

,Region,Subregion,Proposed,Construction,In Development (Proposed + Construction),Shelved,Cancelled,Operating,Idled,Mothballed,Retired
0,Asia,Central Asia,0.0,0.0,0.0,0.0,0.00,0.20,0.0,0.0,0.0
1,,South-eastern Asia,14.5,3.2,17.7,2.0,2.95,58.95,5.6,0.0,20.9
2,,Southern Asia,0.0,12.3,12.3,2.0,41.95,0.00,0.0,0.0,0.0
3,,Western Asia,25.3,58.6,83.9,0.0,22.74,94.30,0.0,6.7,0.0


## Operating capacity by start year

Rows = `ActualStartYear` floored at 1980, columns = `FacilityType`, filtered to
`Fuel = LNG`, the four built statuses, and **`SCOPE`**.

Two caveats carried over from the global notebook: the 1980 floor drops earlier capacity,
so the `Total` row is not total built capacity; and the tab title says "Operating" while
the filter admits idled, mothballed and retired.

In [10]:
built = in_scope(lng)                          # <-- scope
built = built[built["Status"].isin(BUILT_STATUSES) & built["ActualStartYear"].notna()].copy()
built["ActualStartYear"] = built["ActualStartYear"].astype(int)

dropped = built[built["ActualStartYear"] < START_YEAR_FLOOR]["CapacityinMtpa"].sum()
print(f"pre-{START_YEAR_FLOOR} {SCOPE_NAME} capacity excluded by the floor: {dropped:,.1f} mtpa")

by_year = (built[built["ActualStartYear"] >= START_YEAR_FLOOR]
           .pivot_table(index="ActualStartYear", columns="FacilityType",
                        values="CapacityinMtpa", aggfunc="sum")
           .reindex(columns=["import", "export"]).fillna(0.0))
by_year.index.name = "Start year"
by_year.columns = ["Import capacity", "Export capacity"]
by_year = by_year.reindex(range(START_YEAR_FLOOR, int(by_year.index.max()) + 1)).fillna(0.0)
by_year.loc["Total"] = by_year.sum()
start_year = by_year.round(4)
start_year.tail(6)

pre-1980 Asia capacity excluded by the floor: 96.2 mtpa


,Import capacity,Export capacity
Start year,,
2022,29.20,0.00
2023,57.40,3.80
2024,30.40,0.00
2025,29.80,0.00
2026,5.00,0.00
Total,732.47,165.85


## Owner tabs

Ownership is a single string per unit — `"Parent A [60.0%]; Parent B [40.0%]"` — so each
unit's capacity is split across its parents by the stated fractions. Unlabelled parents
split the unclaimed remainder evenly, and units carrying **no** `Parent` at all are
bucketed as `unknown` rather than dropped — the published tabs do the same, and those 12
import units are worth 18.88 mtpa cancelled and 10.51 mtpa proposed.

The published owner tabs were **not** built from the same snapshot as the capacity tabs.
They came from the retired notebook run against the `owners all corrected` export of
2025-10-23; the capacity pivots predate it. Run against that snapshot the export owner tab
reproduces **exactly**, and the import owner tab reproduces exactly on every shared row.

One difference remains and is not method: 17 export / 73 import minority holders appear
only in the published tabs, because the backend `Terminal operators/owners` tabs carry
holders the flat `Parent` column does not. Every parent this notebook produces exists in
the published tab — the gap is one-directional.

In [11]:
PARENT_RE = re.compile(r"^(?P<name>.*?)\s*\[(?P<pct>[\d.]+)\s*%\]$")
UNATTRIBUTED = "unknown"   # 12 import units carry no Parent at all

def parse_parents(s):
    """'A [60%]; B [40%]' -> [('A', 0.6), ('B', 0.4)]. Unlabelled -> whole share."""
    if not isinstance(s, str) or not s.strip():
        return [(UNATTRIBUTED, 1.0)]   # published tabs bucket blank owners, not drop them
    parts = [p.strip() for p in s.split(";") if p.strip()]
    out = []
    for p in parts:
        m = PARENT_RE.match(p)
        if m:
            out.append((m.group("name").strip(), float(m.group("pct")) / 100.0))
        else:
            out.append((p, None))
    unlabelled = [i for i, (_, f) in enumerate(out) if f is None]
    if unlabelled:
        claimed = sum(f for _, f in out if f is not None)
        share = max(0.0, 1.0 - claimed) / len(unlabelled)
        out = [(n, share if f is None else f) for n, f in out]
    return out

def owner_table(facility_type):
    sub = in_scope(lng)                    # <-- scope
    sub = sub[(sub["FacilityType"] == facility_type) & sub["CapacityinMtpa"].notna()]
    rows = []
    for r in sub.itertuples(index=False):
        for name, frac in parse_parents(r.Parent):
            rows.append((name, r.Status, r.CapacityinMtpa * frac))
    df = pd.DataFrame(rows, columns=["Owner", "Status", "CapacityinMtpa"])
    p = (df.pivot_table(index="Owner", columns="Status",
                        values="CapacityinMtpa", aggfunc="sum")
           .reindex(columns=STATUS_ORDER).fillna(0.0))
    return finish_capacity(p, f"LNG {facility_type} capacity by owner", "Owner")

owners = {f"LNG {ft} capacity by owner": owner_table(ft) for ft in ["export", "import"]}
for k, v in owners.items():
    print(f"{k}: {len(v) - 1} owners, total operating {v.loc['Total', 'Operating']:.4f} mtpa")

LNG export capacity by owner: 73 owners, total operating 153.3336 mtpa
LNG import capacity by owner: 353 owners, total operating 781.8796 mtpa


## Capex tabs

Transcribed from the retired Colab notebook (`Summary table python code.ipynb`) that
produced the published capex tabs. All four reproduce **exactly**. Three details are not
obvious and all three are load-bearing:

- **`CostUSDPerMtpa` is derived, not stored.** It is `CostUSD / CapacityinMtpa`, then
  *unconditionally overwritten* wherever `TotKnownTerminalCostsUSD` and the terminal total
  capacity both exist — the source has the `& isna()` guard commented out, so the
  terminal-level ratio wins even when a unit-level one exists. Import runs after export,
  so import wins on the rare terminal carrying both.
- **The cost sample splits on `Offshore`, but the estimate applies on `Floating`.** These
  are two different columns (336 vs 327 rows). Worse, the apply step tests
  `Floating == "yes"` while the column holds `True`/blank — so it **never matches**, the
  offshore regional table is computed and then never used, and every unit is costed at its
  onshore regional rate. This is a bug in the source, faithfully reproduced here because it
  is what produced the published figures. See the note below before reusing it.
- **Override order matters.** Regional estimate, then unit `CostUSD`, then
  `TotKnownTerminalCostsUSD / NumberOfUnits` — the terminal-level override lands *last* and
  so beats a unit-level cost.

The synthetic terminal-total rows from the capacity fallback are **excluded** here — they
are a capacity device, and carrying them into capex would add a second unit-share of
`TotKnownTerminalCostsUSD` on top of the real units' shares.

Regional means are taken per top-level `Region` (Asia/Africa/Americas/Europe/Oceania),
after collapsing each terminal to one datapoint and trimming to the 10th-90th percentile.
Onshore regions with fewer than three points fall back to the global mean; offshore always
uses the global mean.

In [12]:
# The synthetic terminal-total rows are a capacity device only. In capex they would
# add a second unit-share of TotKnownTerminalCostsUSD on top of the real units' shares
# (Commonwealth LNG +1.83 US$ bn, Tabeer LNG +0.25 US$ bn), so capex runs pre-fallback.
cap = lng[~lng["FromTerminalTotal"]].reset_index(drop=True).copy()

# 1. cost per mtpa, derived - then overridden by the terminal-level ratio wherever
#    both parts exist. The source notebook has the "& isna()" guard commented out, so
#    this override is unconditional; import runs second and wins where both apply.
cap["CostUSDPerMtpa"] = cap["CostUSD"] / cap["CapacityinMtpa"]
n_unit = int(cap["CostUSDPerMtpa"].notna().sum())
for cap_col in ["TotExportLNGTerminalCapacityinMtpa", "TotImportLNGTerminalCapacityinMtpa"]:
    m = cap["TotKnownTerminalCostsUSD"].notna() & cap[cap_col].notna()
    cap.loc[m, "CostUSDPerMtpa"] = (cap.loc[m, "TotKnownTerminalCostsUSD"]
                                    / cap.loc[m, cap_col])
print(f"{n_unit} unit-level cost/mtpa -> {int(cap['CostUSDPerMtpa'].notna().sum())} "
      "after the terminal-level override")

# 2. sample selection splits on Offshore (NOT Floating - different columns)
sample_offshore = cap[(cap["Offshore"] == 1) & cap["CostUSDPerMtpa"].notna()]
sample_onshore = cap[cap["Offshore"].isnull() & cap["CostUSDPerMtpa"].notna()]

def collapse_to_terminal(frame):
    """One datapoint per terminal, carrying that terminal's mean cost-per-mtpa."""
    avg = frame.groupby("ProjectID")["CostUSDPerMtpa"].mean()
    out = frame.drop_duplicates(subset=["ProjectID"], keep="first").copy()
    out["CostUSDPerMtpa"] = out["ProjectID"].map(avg)
    return out

def trim(frame):
    lo = frame["CostUSDPerMtpa"].quantile(CAPEX_QLO)
    hi = frame["CostUSDPerMtpa"].quantile(CAPEX_QHI)
    return frame[frame["CostUSDPerMtpa"].between(lo, hi, inclusive="both")]

sample_offshore = trim(collapse_to_terminal(sample_offshore))
sample_onshore = trim(collapse_to_terminal(sample_onshore))
print(f"cost sample after collapse + {CAPEX_QLO:.0%}-{CAPEX_QHI:.0%} trim: "
      f"{len(sample_onshore)} onshore, {len(sample_offshore)} offshore terminals")

REGIONS = [r for r in cap["Region"].dropna().unique() if r != "--"]

def regional_costs(sample, facility_type, always_global):
    """Mean cost-per-mtpa per Region; global mean where thin (or always, offshore)."""
    g = sample[sample["FacilityType"] == facility_type]
    t = pd.DataFrame(index=REGIONS)
    t["n"] = g.groupby("Region")["CostUSDPerMtpa"].count()
    t["cost"] = g.groupby("Region")["CostUSDPerMtpa"].mean()
    global_mean = g["CostUSDPerMtpa"].mean()
    if always_global:
        t["cost"] = global_mean
    else:
        t.loc[(t["n"] < CAPEX_MIN_POINTS) | (t["n"].isnull()), "cost"] = global_mean
    return t

471 unit-level cost/mtpa -> 673 after the terminal-level override


cost sample after collapse + 10%-90% trim: 212 onshore, 88 offshore terminals


### How thin is the cost sample underneath these capex figures?

Every capex cell is either a reported cost or `capacity x a regional rate`. The rate is a
mean over the trimmed sample below, so the row for `Asia` is what the Asia capex tabs
actually rest on. Import is well supported; **export is not** — read the counts before
quoting an export capex number.

In [13]:
for ft in ["import", "export"]:
    g = sample_onshore[sample_onshore["FacilityType"] == ft]
    t = pd.DataFrame({"terminals": g.groupby("Region")["CostUSDPerMtpa"].count(),
                      "mean US$/mtpa": g.groupby("Region")["CostUSDPerMtpa"].mean()})
    t["mean US$/mtpa"] = t["mean US$/mtpa"].map(lambda v: f"{v:,.0f}")
    print(f"=== {ft}: trimmed onshore cost sample by Region ===")
    print(t.to_string())
    n_asia = int(g.groupby("Region")["CostUSDPerMtpa"].count().get("Asia", 0))
    print(f"  global mean {g['CostUSDPerMtpa'].mean():,.0f} (n={len(g)});  Asia n={n_asia}"
          f" -> {'uses the Asia mean' if n_asia >= CAPEX_MIN_POINTS else 'FALLS BACK to global'}")
    print()

# Exclude the synthetic terminal-total rows - they copy a cost in and would inflate this.
_real = lng[~lng["FromTerminalTotal"]]
print("Cost coverage in this download (real LNG unit rows, world):")
for c in ["CostUSD", "TotKnownTerminalCostsUSD"]:
    print(f"  {c:<28s} {_real[c].notna().sum():>5d}")
print("  For comparison, the global folder README records the gem-db-ops pull at")
print("  CostUSD 251 / TotKnown 335, and the 2025 release download at 437 / 613.")
print("  This all-fields download is better covered than either.")

=== import: trimmed onshore cost sample by Region ===
          terminals mean US$/mtpa
Region                           
Africa            2   331,928,155
Americas         11   392,386,182
Asia            113   280,928,023
Europe           19   306,577,911
  global mean 293,447,940 (n=145);  Asia n=113 -> uses the Asia mean

=== export: trimmed onshore cost sample by Region ===
          terminals mean US$/mtpa
Region                           
Africa            9   415,055,408
Americas         37   735,315,862
Asia              9   737,821,924
Europe            9   885,878,651
Oceania           3   675,353,188
  global mean 710,172,392 (n=67);  Asia n=9 -> uses the Asia mean

Cost coverage in this download (real LNG unit rows, world):
  CostUSD                        493
  TotKnownTerminalCostsUSD       692
  For comparison, the global folder README records the gem-db-ops pull at
  CostUSD 251 / TotKnown 335, and the 2025 release download at 437 / 613.
  This all-fields download is b

In [14]:
def costed_units(facility_type, token=None, always_global=None):
    """Per-unit CostUSDTotal: regional estimate, then unit cost, then terminal cost.

    token / always_global default to the configured values. They are parameters only so
    the comparison cell below can price the same units both ways.
    """
    token = FLOATING_TRUE_TOKEN if token is None else token
    always_global = (CAPEX_FLOATING_ALWAYS_GLOBAL if always_global is None
                     else always_global)
    onshore_rates = regional_costs(sample_onshore, facility_type, always_global=False)
    offshore_rates = regional_costs(sample_offshore, facility_type,
                                    always_global=always_global)

    d = cap[cap["FacilityType"] == facility_type].reset_index(drop=True).copy()
    d["NumberOfUnits"] = d.groupby("ProjectID")["UnitID"].transform("nunique")
    d["CostUSDTotal"] = np.nan

    for region in REGIONS:
        # With CAPEX_FIX_FLOATING the token is True and this branch fires on FSRUs;
        # with it False the token is "yes", nothing matches, and every unit costs onshore.
        is_floating = (d["Floating"] == token) & (d["Region"] == region)
        d.loc[is_floating, "CostUSDTotal"] = (d.loc[is_floating, "CapacityinMtpa"]
                                              * offshore_rates.loc[region, "cost"])
        rest = (d["Floating"] != token) & (d["Region"] == region)
        d.loc[rest, "CostUSDTotal"] = (d.loc[rest, "CapacityinMtpa"]
                                       * onshore_rates.loc[region, "cost"])

    m = d["CostUSD"].notna()
    d.loc[m, "CostUSDTotal"] = d.loc[m, "CostUSD"]
    # terminal-level cost lands last, so it beats a unit-level CostUSD
    m = d["TotKnownTerminalCostsUSD"].notna()
    d.loc[m, "CostUSDTotal"] = (d.loc[m, "TotKnownTerminalCostsUSD"]
                                / d.loc[m, "NumberOfUnits"])
    return d

def capex_table(facility_type, geo_col, tab, index_name):
    # costed_units() runs on the FULL global frame so the regional rates above are
    # unchanged; scope lands only on the finished per-unit costs.
    d = in_scope(costed_units(facility_type))          # <-- scope
    p = (d.pivot_table(index=geo_col, columns="Status",
                       values="CostUSDTotal", aggfunc="sum")
           .reindex(columns=STATUS_ORDER).fillna(0.0)) / 1e9
    return finish_capacity(p, tab, index_name)

capex = {}
for ft in ["export", "import"]:
    capex[f"LNG {ft} terminal capex by country/area"] = capex_table(
        ft, "Country/Area", f"LNG {ft} terminal capex by country/area", "Country/Area")
    capex[f"LNG {ft} terminal capex by region"] = capex_table(
        ft, "SubRegion", f"LNG {ft} terminal capex by region", "Subregion")
for k, v in capex.items():
    print(f"{k}: total {v.loc['Total'].sum():,.1f} US$ bn across all statuses")

LNG export terminal capex by country/area: total 358.3 US$ bn across all statuses
LNG export terminal capex by region: total 358.3 US$ bn across all statuses
LNG import terminal capex by country/area: total 474.4 US$ bn across all statuses
LNG import terminal capex by region: total 474.4 US$ bn across all statuses


### The floating-unit fix, priced both ways

`CAPEX_FIX_FLOATING` is on in this cut, so FSRUs cost at the offshore rate. The published
global tabs have it off. This cell prices the same units both ways so the difference is on
the record rather than buried in a config flag.

Three scenarios: **A** reproduces the published defect (everything onshore); **B** fixes
only the token, so floating units take the *global* offshore mean; **C** also lets offshore
use *regional* means, the way onshore already does. C is what this notebook runs.

In [15]:
SCENARIOS = [
    ("A  published defect (all onshore)", "yes", True),
    ("B  token fixed -> global offshore", True,  True),
    ("C  token + regional offshore     ", True,  False),
]

def scenario_capex(token, always_global, subregions=None, statuses=("proposed", "construction")):
    out = {}
    for ft in ["import", "export"]:
        d = costed_units(ft, token=token, always_global=always_global)
        d = in_scope(d) if subregions is None else d[d["SubRegion"].isin(subregions)]
        out[ft] = d[d["Status"].isin(statuses)]["CostUSDTotal"].sum() / 1e9
    return out

for label, subs in [(f"{SCOPE_NAME} (all subregions)", None),
                    ("South + Southeast Asia", SSEA_SUBREGIONS)]:
    print("=" * 74)
    print(f"{label} - IN-DEVELOPMENT capex, US$ bn")
    print("=" * 74)
    print(f"{'scenario':<36s}{'import':>10s}{'export':>10s}{'total':>10s}")
    base = None
    for name, tok, ag in SCENARIOS:
        v = scenario_capex(tok, ag, subs)
        tot = v["import"] + v["export"]
        base = tot if base is None else base
        delta = "" if name.startswith("A") else f"   {tot - base:+.1f}"
        print(f"{name:<36s}{v['import']:>10.1f}{v['export']:>10.1f}{tot:>10.1f}{delta}")
    print()

# Why export barely moves: the estimate only prices units with no reported cost.
print("Units the fix actually reprices (floating AND no reported cost):")
for ft in ["import", "export"]:
    d = in_scope(lng[~lng["FromTerminalTotal"]])
    d = d[d["FacilityType"] == ft]
    aff = d["Floating"].notna() & d["CostUSD"].isna() & d["TotKnownTerminalCostsUSD"].isna()
    print(f"  {SCOPE_NAME} {ft:<7s} {int(aff.sum()):>3d} units, "
          f"{d.loc[aff, 'CapacityinMtpa'].sum():>7.1f} mtpa")
print("  Everything else either is not floating, or carries a reported cost that")
print("  overrides the regional estimate entirely.")

Asia (all subregions) - IN-DEVELOPMENT capex, US$ bn
scenario                                import    export     total


A  published defect (all onshore)        116.1     102.8     218.9
B  token fixed -> global offshore        110.5     102.8     213.3   -5.6


C  token + regional offshore             108.7     102.8     211.5   -7.4

South + Southeast Asia - IN-DEVELOPMENT capex, US$ bn
scenario                                import    export     total
A  published defect (all onshore)         37.0      54.3      91.3


B  token fixed -> global offshore         32.9      54.3      87.2   -4.1


C  token + regional offshore              31.6      54.3      85.9   -5.4

Units the fix actually reprices (floating AND no reported cost):
  Asia import   70 units,   214.9 mtpa
  Asia export    3 units,     2.7 mtpa
  Everything else either is not floating, or carries a reported cost that
  overrides the regional estimate entirely.


## Validation against the published tabs — optional

Inert until `REFERENCE_JSON` exists. Point it at the published 2026 tabs once they are up
and re-run to diff every computed tab cell-by-cell; until then these cells no-op.

The 2025 run is the evidence the method is right. Do not treat a skipped validation as a
passed one.

In [16]:
# The capacity and start-year tabs were pasted as values rounded to 1 dp, so a
# 0.05 residual there is display rounding, not a method difference. The owner and
# capex tabs carry full precision.
TOL_ROUNDED = 0.06   # mtpa, tabs stored at 1 dp
TOL_EXACT = 0.001    # mtpa or US$ bn, tabs stored at full precision

HAVE_REFERENCE = REFERENCE_JSON.exists()
published = json.loads(REFERENCE_JSON.read_text()) if HAVE_REFERENCE else {}
if not HAVE_REFERENCE:
    print(f"no published reference at {REFERENCE_JSON} - validation skipped.")

def published_frame(tab):
    """Published rows -> DataFrame keyed on the row label, numeric cells only."""
    rows = published[tab]
    hdr_i = next(i for i, r in enumerate(rows)
                 if r and any(str(c).strip().lower().startswith(
                     ("proposed", "import capacity")) for c in r))
    header = [str(c).strip() for c in rows[hdr_i]]
    # On region tabs the label lives in column 1; column 0 (Region) is blank on
    # every continuation row, so keying the filter on column 0 drops most of the tab.
    label_i = 1 if header[0] == "Region" else 0
    body = [r for r in rows[hdr_i + 1:]
            if r and (str(r[label_i]).strip() if len(r) > label_i else "")
            or (r and label_i == 1 and str(r[0]).strip())]
    recs = {}
    for r in body:
        r = list(r) + [""] * (len(header) - len(r))
        label = str(r[label_i]).strip()
        if not label and label_i == 1:
            label = str(r[0]).strip()   # the Total row carries no subregion
        vals = r[label_i + 1:]
        names = header[label_i + 1:]
        recs[label] = {n: (float(v) if str(v).strip() not in ("", "None") else 0.0)
                       for n, v in zip(names, vals)}
    return pd.DataFrame(recs).T

def validate(tab, computed, label_col=None, tol=TOL_EXACT):
    pub = published_frame(tab)
    comp = computed.copy()
    if label_col:      # region tabs were flattened to a plain frame
        comp = comp.set_index(label_col)
        comp = comp.drop(columns=["Region"], errors="ignore")
    comp.index = [str(i).strip() for i in comp.index]
    # align on shared rows / columns by position of the numeric block
    shared_rows = [r for r in comp.index if r in pub.index]
    missing = [r for r in pub.index if r not in comp.index]
    extra = [r for r in comp.index if r not in pub.index]
    n = min(comp.shape[1], pub.shape[1])
    diffs = []
    for r in shared_rows:
        for j in range(n):
            a = float(comp.loc[r].iloc[j]); b = float(pub.loc[r].iloc[j])
            if abs(a - b) > tol:
                diffs.append((r, str(comp.columns[j]), round(a, 3), round(b, 3), round(a - b, 3)))
    print(f"\n=== {tab}")
    print(f"    rows: {len(shared_rows)} shared, {len(missing)} only-published, {len(extra)} only-computed")
    if missing[:6]: print(f"    only in published: {missing[:6]}")
    if extra[:6]:   print(f"    only in computed:  {extra[:6]}")
    if not diffs:
        print(f"    MATCH - no cell differs by more than {tol}")
    else:
        print(f"    {len(diffs)} cell(s) over tolerance:")
        for d in diffs[:12]:
            print(f"      {d[0]:<28s} {d[1]:<40s} computed {d[2]:>10} vs published {d[3]:>10}  ({d[4]:+})")
        if len(diffs) > 12: print(f"      ... and {len(diffs) - 12} more")
    return diffs

if HAVE_REFERENCE:
    results = {}
    for tab in ["LNG export capacity by country/area", "LNG import capacity by country/area"]:
        results[tab] = validate(tab, capacity[tab], tol=TOL_ROUNDED)
    for tab in ["LNG export capacity by region", "LNG import capacity by region"]:
        results[tab] = validate(tab, region_tabs[tab], label_col="Subregion", tol=TOL_ROUNDED)
    results["Operating LNG capacity by start year"] = validate(
        "Operating LNG capacity by start year",
        start_year.rename(index=lambda x: str(x)), tol=TOL_ROUNDED)
    for tab, frame in owners.items():
        results[tab] = validate(tab, frame, tol=TOL_EXACT)
    for tab, frame in capex.items():
        results[tab] = validate(tab, frame, tol=TOL_EXACT)

no published reference at published-2026-q4-reference.json - validation skipped.


In [17]:
if not HAVE_REFERENCE:
    print('validation skipped - no published reference for this release yet')
else:
    print("\n" + "=" * 72)
    print(f"{'tab':<46s} {'cells over tol':>14s}")
    print("=" * 72)
    for tab, d in results.items():
        flag = "OK" if not d else str(len(d))
        print(f"{tab:<46s} {flag:>14s}")
    print("=" * 72)
    print("""
    Expected on the September snapshot - the one the published tabs were built from,
    apart from the two owner tabs:
      4 capacity tabs   2-3 each, all published-side (stale In Development / Total cells)
      start year        0
      4 capex tabs      0 - exact
      2 owner tabs      27 / 9 - these tabs came from the October snapshot; see the
                        Oct2025 notebook, where both go to 0
    """)

validation skipped - no published reference for this release yet


### The published tabs do not reconcile against themselves

With the terminal-level fallback in place and no hand-entered deltas, every remaining
capacity difference is a published-side error. Three checks, using only published cells:

1. **`In Development` != `Proposed` + `Construction`** on Germany (+5.0), Western Europe
   (+5.1), Northern America (-10.5), France (+0.2) and the Netherlands (+0.1). The
   proposed cells were edited and the derived cells never recomputed.
2. **The `Total` row != the sum of its own data rows**, in every one of the four capacity
   tabs. Totals were computed at one moment and cells edited afterwards.
3. **The region tab != the country tab**, for the same underlying data. Northern America
   is 0.9 higher in the region tab than its own United States + Canada cells;
   South-eastern Asia is 2.0 *lower* than its member countries.

Check 3 explains the last two residuals. The Malaysia +2.0 is a hand edit applied to the
**country tab only** - the published region tab agrees with this notebook. The Northern
America +0.9 is the reverse, a region-tab-only edit. In both cases the reproduction sits
on the side that reconciles.

None of the three can happen here: `In Development` and `Total` are derived, and both
geographies roll up from the same unit rows. All three are worth raising with the tracker
lead before the Q4 2026 run.

In [18]:
def audit_published(tab):
    f = published_frame(tab)
    in_dev = (f.iloc[:, 2] - (f.iloc[:, 0] + f.iloc[:, 1])).round(3)
    bad_dev = in_dev[in_dev.abs() > 0.05]

    body = f.drop(index="Total", errors="ignore")
    bad_tot = {}
    if "Total" in f.index:
        d = (f.loc["Total"] - body.sum()).round(3)
        bad_tot = {k: v for k, v in d.items() if abs(v) > 0.05}

    print(f"\n=== {tab}")
    if len(bad_dev):
        print("    In Development != Proposed + Construction:")
        for k, v in bad_dev.items():
            print(f"      {k:<40s} {v:+.2f}")
    else:
        print("    In Development reconciles on every row")
    if bad_tot:
        print("    Total row != sum of data rows:")
        for k, v in bad_tot.items():
            print(f"      {k:<40s} {v:+.2f}")
    else:
        print("    Total row reconciles")

if HAVE_REFERENCE:
    for tab in ["LNG export capacity by country/area", "LNG import capacity by country/area",
                "LNG export capacity by region", "LNG import capacity by region"]:
        audit_published(tab)


# Third check: does the published region tab agree with the published country tab?
COUNTRY_TO_SUBREGION = (lng[["Country/Area", "SubRegion"]].dropna().drop_duplicates()
                           .set_index("Country/Area")["SubRegion"].to_dict())

def audit_region_vs_country(ft):
    reg = published_frame(f"LNG {ft} capacity by region").drop(index="Total", errors="ignore")
    cty = published_frame(f"LNG {ft} capacity by country/area").drop(index="Total", errors="ignore")
    rolled = cty.groupby(cty.index.map(COUNTRY_TO_SUBREGION)).sum()
    shared = [r for r in reg.index if r in rolled.index]
    print(f"\n=== LNG {ft}: region tab vs its own country tab")
    bad = False
    for r in shared:
        d = (reg.loc[r] - rolled.loc[r]).round(3)
        for col, v in d.items():
            if abs(v) > 0.06:
                print(f"    {r:<30s} {col:<42s} {v:+.2f}")
                bad = True
    if not bad:
        print("    every subregion agrees with the sum of its countries")

if HAVE_REFERENCE:
    for ft in ["export", "import"]:
        audit_region_vs_country(ft)
else:
    print("published self-consistency audit skipped - no reference workbook yet")

published self-consistency audit skipped - no reference workbook yet


## South and Southeast Asia — the write-up's figures

The Asia tabs above cover all five UN M49 Asian subregions. The write-up is about two of
them: **Southern Asia** and **South-eastern Asia**. This section cuts exactly that pair
and prints the specific claims the draft makes, so each can be checked against a current
number rather than the September 2025 one it was written from.

In [19]:
def ssea_table(facility_type):
    sub = lng[lng["SubRegion"].isin(SSEA_SUBREGIONS)]
    sub = sub[sub["FacilityType"] == facility_type]
    p = (sub.pivot_table(index="SubRegion", columns="Status",
                         values="CapacityinMtpa", aggfunc="sum")
           .reindex(columns=STATUS_ORDER).fillna(0.0))
    return finish_capacity(p, f"S+SE Asia {facility_type}", "Subregion")

ssea_import = ssea_table("import")
ssea_export = ssea_table("export")

def ssea_capex_table():
    frames = []
    for ft in ["import", "export"]:
        d = costed_units(ft)                      # global rates, as everywhere else
        d = d[d["SubRegion"].isin(SSEA_SUBREGIONS)]
        p = (d.pivot_table(index="SubRegion", columns="Status",
                           values="CostUSDTotal", aggfunc="sum")
               .reindex(columns=STATUS_ORDER).fillna(0.0)) / 1e9
        t = finish_capacity(p, f"S+SE capex {ft}", "Subregion")
        t.index = [f"{i} ({ft})" for i in t.index]
        frames.append(t)
    return pd.concat(frames)

ssea_capex = ssea_capex_table()

print("=" * 78)
print("SOUTH + SOUTHEAST ASIA - LNG IMPORT CAPACITY (mtpa)")
print("=" * 78)
print(ssea_import.round(1).to_string())
print()
print("=" * 78)
print("SOUTH + SOUTHEAST ASIA - LNG EXPORT CAPACITY (mtpa)")
print("=" * 78)
print(ssea_export.round(1).to_string())
print()
print("=" * 78)
print("SOUTH + SOUTHEAST ASIA - LNG TERMINAL CAPEX (US$ bn, mostly ESTIMATED)")
print("=" * 78)
print(ssea_capex.round(1).to_string())

SOUTH + SOUTHEAST ASIA - LNG IMPORT CAPACITY (mtpa)
                    Proposed  Construction  In Development (Proposed + Construction)  Shelved  Cancelled  Operating  Idled  Mothballed  Retired
Subregion                                                                                                                                      
South-eastern Asia      51.8          18.1                                      69.9     23.2       55.0       62.6    0.0         0.0      0.1
Southern Asia           89.9          11.0                                     100.9     12.6      107.4       75.9    0.0         0.0      0.0
Total                  141.7          29.1                                     170.8     35.8      162.3      138.5    0.0         0.0      0.1

SOUTH + SOUTHEAST ASIA - LNG EXPORT CAPACITY (mtpa)
                    Proposed  Construction  In Development (Proposed + Construction)  Shelved  Cancelled  Operating  Idled  Mothballed  Retired
Subregion                      

### Checking the draft's claims

Each block prints the figure the draft asserts next to what this snapshot says. The draft
was written from a September 2025 pull; these come from the 2026-08-28 download, so
movement is expected and is the point of re-running.

In [20]:
T = ssea_import.loc["Total"]
shelved_cancelled = T["Shelved"] + T["Cancelled"]
operating = T["Operating"]
in_dev = T[IN_DEV]

print("=" * 78)
print("DRAFT CLAIM 1: 'shelved or cancelled a combined 169.2 mtpa of proposed LNG import")
print("               capacity - more than the 133.9 mtpa currently operating'")
print("-" * 78)
print(f"  shelved + cancelled   drafted 169.2   now {shelved_cancelled:>7.1f} mtpa")
print(f"  operating             drafted 133.9   now {operating:>7.1f} mtpa")
print(f"  -> 'more than operating' still holds: {shelved_cancelled > operating} "
      f"({shelved_cancelled / operating:.2f}x)")

print()
print("=" * 78)
print("DRAFT CLAIM 2: 'In South Asia, cancelled capacity alone exceeded operating capacity'")
print("-" * 78)
s = ssea_import.loc["Southern Asia"]
print(f"  Southern Asia   cancelled {s['Cancelled']:>7.1f}   operating {s['Operating']:>7.1f} mtpa")
print(f"  -> holds: {s['Cancelled'] > s['Operating']} ({s['Cancelled'] / s['Operating']:.2f}x)")

print()
print("=" * 78)
print("DRAFT CLAIM 3: 'in Southeast Asia shelved and cancelled capacity roughly matched")
print("               operating capacity'")
print("-" * 78)
e = ssea_import.loc["South-eastern Asia"]
sc = e["Shelved"] + e["Cancelled"]
print(f"  SE Asia   shelved+cancelled {sc:>7.1f}   operating {e['Operating']:>7.1f} mtpa")
print(f"  -> ratio {sc / e['Operating']:.2f}x")
print(f"  -> 'roughly matched' is a stretch at {sc / e['Operating']:.2f}x; "
      f"'exceeded' / 'around {sc / e['Operating']:.0%} of' is the accurate phrasing.")

print()
print("=" * 78)
print("DRAFT CLAIM 4: 'more than US$ 200 billion of planned LNG import and gas power")
print("               infrastructure exposed'")
print("-" * 78)
imp_dev = ssea_capex.loc[[i for i in ssea_capex.index
                          if "(import)" in i and "Total" not in i], IN_DEV].sum()
exp_dev = ssea_capex.loc[[i for i in ssea_capex.index
                          if "(export)" in i and "Total" not in i], IN_DEV].sum()
print(f"  S+SE Asia LNG IMPORT capex, in development   {imp_dev:>8.1f} US$ bn  (estimated)")
print(f"  S+SE Asia LNG EXPORT capex, in development   {exp_dev:>8.1f} US$ bn  (estimated)")
print(f"  LNG subtotal                                 {imp_dev + exp_dev:>8.1f} US$ bn")
print()
print("  NOTE: the US$200bn claim spans LNG import terminals AND gas power plants.")
print("  The gas-power half is GOGPT, which is NOT in this dataset - it has to come")
print("  from a separate pull. The LNG half is the figure above, and it is an ESTIMATE:")
print("  most units carry no reported cost and are priced at capacity x a regional rate.")

print()
print("=" * 78)
print("Extra context for the piece")
print("-" * 78)
print(f"  S+SE Asia import in development (proposed + construction)  {in_dev:>7.1f} mtpa")
print(f"    of which proposed      {T['Proposed']:>7.1f} mtpa")
print(f"    of which construction  {T['Construction']:>7.1f} mtpa")

DRAFT CLAIM 1: 'shelved or cancelled a combined 169.2 mtpa of proposed LNG import
               capacity - more than the 133.9 mtpa currently operating'
------------------------------------------------------------------------------
  shelved + cancelled   drafted 169.2   now   198.2 mtpa
  operating             drafted 133.9   now   138.5 mtpa
  -> 'more than operating' still holds: True (1.43x)

DRAFT CLAIM 2: 'In South Asia, cancelled capacity alone exceeded operating capacity'
------------------------------------------------------------------------------
  Southern Asia   cancelled   107.4   operating    75.9 mtpa
  -> holds: True (1.41x)

DRAFT CLAIM 3: 'in Southeast Asia shelved and cancelled capacity roughly matched
               operating capacity'
------------------------------------------------------------------------------
  SE Asia   shelved+cancelled    78.2   operating    62.6 mtpa
  -> ratio 1.25x
  -> 'roughly matched' is a stretch at 1.25x; 'exceeded' / 'around 125% o

## Write the workbook

One sheet per published tab, same order as the published workbook. Output is a gitignored
`.xlsx` — this is a validation artifact, not a deliverable.

In [21]:
TAB_ORDER = [
    ("LNG export capacity by country/area", capacity["LNG export capacity by country/area"], True),
    ("LNG import capacity by country/area", capacity["LNG import capacity by country/area"], True),
    ("LNG export capacity by region", region_tabs["LNG export capacity by region"], False),
    ("LNG import capacity by region", region_tabs["LNG import capacity by region"], False),
    ("LNG export capacity by owner", owners["LNG export capacity by owner"], True),
    ("LNG import capacity by owner", owners["LNG import capacity by owner"], True),
    ("Operating LNG capacity by start year", start_year, True),
    ("LNG export terminal capex by country/area", capex["LNG export terminal capex by country/area"], True),
    ("LNG import terminal capex by country/area", capex["LNG import terminal capex by country/area"], True),
    ("LNG export terminal capex by region", capex["LNG export terminal capex by region"], True),
    ("LNG import terminal capex by region", capex["LNG import terminal capex by region"], True),
]

def sheet_name(tab):
    """Excel forbids / \ ? * [ ] : in sheet names and caps them at 31 chars."""
    safe = re.sub(r"[/\?*\[\]:]", "-", tab)
    return safe[:31]

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as xl:
    for name, frame, with_index in TAB_ORDER:
        frame.to_excel(xl, sheet_name=sheet_name(name), index=with_index)
    ssea_import.to_excel(xl, sheet_name="S+SE Asia import")
    ssea_export.to_excel(xl, sheet_name="S+SE Asia export")
    ssea_capex.to_excel(xl, sheet_name="S+SE Asia capex")

print(f"wrote {OUTPUT_XLSX} ({len(TAB_ORDER) + 3} sheets, scope = {SCOPE_NAME})")

wrote GGIT-LNG-summary-sheets-Asia-2026Q4.xlsx (14 sheets, scope = Asia)


## Landing-page stats

The handful of figures that get quoted in the release write-up.

In [22]:
ec = capacity["LNG export capacity by country/area"].loc["Total"]
ic = capacity["LNG import capacity by country/area"].loc["Total"]
scoped = in_scope(lng)
print(f"{RELEASE_LABEL} LNG terminals\n")
print(f"  export operating       {ec['Operating']:>9,.1f} mtpa")
print(f"  export in development  {ec[IN_DEV]:>9,.1f} mtpa")
print(f"  import operating       {ic['Operating']:>9,.1f} mtpa")
print(f"  import in development  {ic[IN_DEV]:>9,.1f} mtpa")
print(f"\n  terminals (unique)     {scoped['ProjectID'].nunique():>9,}")
print(f"  units                  {len(scoped):>9,}")
print(f"  countries/areas        {scoped['Country/Area'].nunique():>9,}")

Q4 2026 — Asia LNG terminals

  export operating           153.4 mtpa
  export in development      113.9 mtpa
  import operating           782.0 mtpa
  import in development      492.3 mtpa

  terminals (unique)           361
  units                        528
  countries/areas               34


## Notes on this cut

1. **Scope lands at pivot time, on purpose.** The capex regional rates are means over a
   global sample of terminals, trimmed to the 10th–90th percentile and falling back to a
   global mean below three points. Filtering the input to Asia first would refit those
   rates on a fraction of the sample and move every estimated cell. If you ever want
   genuinely Asia-fitted rates that is a different — and defensible — table, but it is not
   what this notebook produces, and it would not be comparable with the published global
   tabs.
2. **This cut departs from the published method in one deliberate place**: the floating
   branch. See note 3.
3. **The floating-unit defect is FIXED here, unlike the published tabs.**
   `CAPEX_FIX_FLOATING = True` means FSRUs cost at the offshore rate instead of the
   onshore one. That is the right call for an FSRU-heavy region — but it makes these
   capex figures **not directly comparable** with the published global tabs, and that has
   to be said wherever they are quoted. Set the flag to `False` to reproduce the
   published behaviour; the comparison cell prices both ways either way.
   The other transcribed defect, the unconditional terminal-level `CostUSDPerMtpa`
   override, is **still in place** — it is not obviously wrong, just undocumented.
4. **Capex is an estimate, not a reported total**, and the export side is thin. Every
   cell is either a reported cost or capacity × a regional rate. The rate for Asia rests
   on **113 terminals for import but only 9 for export** (see the cost-sample diagnostic),
   so Asian export capex is the least reliable number in this workbook — quote it with a
   range, or not at all.
5. **The cost-coverage question in the global folder's README is answered, and the answer
   is good news.** That README flags the `gem-db-ops` pull as carrying far fewer cost
   values than the 2025 release download (`CostUSD` 251 vs 437, `TotKnownTerminalCostsUSD`
   335 vs 613) and asks whether the database lost them. This all-fields download carries
   **493 and 692** — more than either. So the database did not lose anything;
   `lng/pull.py` is mapping a narrower field. Worth fixing there, and worth preferring the
   all-fields download for capex work until it is.
6. **Validation is inert.** There are no published Asia tabs to diff against. The evidence
   the method is right is the 2025 global reproduction, not anything in this notebook.
7. `Total` rows here are **Asia** totals, not world totals.